# CROPS

`CROPS` (Changepoints for a Range Of PenaltieS) runs `PELT` across a range of penalties and returns the full solution path, then selects one segmentation from that path using BIC or an elbow criterion. It is the tool to reach for when you don't yet know a good penalty value, or when you want to inspect how the segmentation changes with the penalty.

## Basic usage

`CROPS` shares its objective and its `cost` argument with [PELT](pelt.ipynb). Instead of a single `penalty`, you supply a search range through `min_penalty` and `max_penalty`. If both are left as `None`, sensible defaults are derived from the cost.

In [ ]:
import plotly.io as pio

from skchange.datasets import generate_piecewise_normal_data
from skchange.detectors import CROPS
from skchange.interval_scorers import L2Cost
from skchange.utils.plotting import plot_detections

pio.renderers.default = "notebook"

X = generate_piecewise_normal_data(
    means=[0, 10, 0, -3, 5, 1],
    lengths=[30, 5, 15, 50, 60, 40],
    seed=0,
)

detector = CROPS(L2Cost(), selection_method="bic")
detector.fit(X)
result = detector.predict_all(X)

print("Selected penalty:", result["optimal_penalty"])
print("Changepoints:", result["changepoints"])

plot_detections(X, changepoints=result["changepoints"]).show()

## Inspecting the solution path

`predict_all` also returns the segmentation for every number of changepoints along the path. This is useful for visualising the classical *penalty vs. number of changepoints* diagnostic plot, or for comparing candidate segmentations side by side.

In [ ]:
metadata = result["changepoints_metadata"]
for n_cp, penalty, bic in zip(
    metadata["num_changepoints"],
    metadata["penalty"],
    metadata["bic_value"],
):
    print(f"n_changepoints={n_cp:>2d}  penalty={penalty:>8.3f}  bic={bic:>9.3f}")

## Parameters worth knowing

- `cost`: Any interval scorer of type `cost`. Same role as in `PELT`.
- `min_penalty`, `max_penalty`: Lower and upper limits for the penalty search. `None` picks sensible defaults from the cost.
- `selection_method`: Either `"bic"` (default) or `"elbow"`. Chooses which segmentation on the path to return through `predict`.
- The rest of the parameters (`min_segment_length`, `step_size`, `prune`, ...) behave exactly like in `PELT`.

See the full API reference for [CROPS](../../api_reference/auto_generated/skchange.detectors.CROPS.rst).

## See also

- [PELT](pelt.ipynb): A single exact segmentation at a fixed penalty.
- [Penalties](../concepts/penalties.ipynb): Background on how the penalty controls the number of segments.